# K-Nearest Neighbors from Scratch — Iris Dataset

A from-scratch KNN classifier (pure Python/NumPy, no sklearn), benchmarked against scikit-learn's `KNeighborsClassifier` across different values of K and distance metrics.


In [4]:
import urllib.request
import random
import math
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
import numpy as np


In [5]:
iris = load_iris()
X = iris.data 
y = iris.target

In [6]:
np.random.seed(42)
indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

## Train / Test Split

In [7]:
split_idx = int(0.8 * len(X))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(X_train.shape, X_test.shape)

(120, 4) (30, 4)


## Min-Max Scaling

In [8]:
X_train_min = X_train.min(axis=0)
X_train_max = X_train.max(axis=0)

X_train_norm = (X_train - X_train_min) / (X_train_max - X_train_min)
X_test_norm = (X_test - X_train_min) / (X_train_max - X_train_min)

In [9]:
def euclidean_distance(point1, point2):
    return np.sqrt(np.sum((point1 - point2) ** 2))

In [10]:
def knn_predict(X_train, y_train, test_point, k):
    distances = []
    for i in range(len(X_train)):
        dist = euclidean_distance(X_train[i], test_point)
        distances.append((dist, y_train[i]))
    
    distances.sort(key=lambda x: x[0])
    
    neighbors = distances[:k]
    neighbor_classes = [neighbor[1] for neighbor in neighbors]
    
    majority_vote = Counter(neighbor_classes).most_common(1)[0][0]
    return majority_vote

In [12]:
k_values = [1, 3, 5, 7, 11]
test_accuracies = []
train_accuracies = []

print(" KNN results :")
print("-" * 60)
for k in k_values:
    correct_predictions_test = 0
    for i in range(len(X_test_norm)):
        prediction = knn_predict(X_train_norm, y_train, X_test_norm[i], k)
        if prediction == y_test[i]:
            correct_predictions_test += 1
            
    test_accuracy = correct_predictions_test / len(y_test)
    test_accuracies.append(test_accuracy)
    
    correct_predictions_train = 0
    for i in range(len(X_train_norm)):
        prediction = knn_predict(X_train_norm, y_train, X_train_norm[i], k)
        if prediction == y_train[i]:
            correct_predictions_train += 1
            
    train_accuracy = correct_predictions_train / len(y_train)
    train_accuracies.append(train_accuracy)
    
    print(f"Accuracy for K = {k:2d}  -->  {test_accuracy * 100:.2f}%")
    print(f"Accuracy(train) for K = {k:2d}  -->  {train_accuracy * 100:.2f}%")
    print("-" * 40)

 KNN results :
------------------------------------------------------------
Accuracy for K =  1  -->  96.67%
Accuracy(train) for K =  1  -->  100.00%
----------------------------------------
Accuracy for K =  3  -->  93.33%
Accuracy(train) for K =  3  -->  96.67%
----------------------------------------
Accuracy for K =  5  -->  93.33%
Accuracy(train) for K =  5  -->  96.67%
----------------------------------------
Accuracy for K =  7  -->  93.33%
Accuracy(train) for K =  7  -->  96.67%
----------------------------------------
Accuracy for K = 11  -->  96.67%
Accuracy(train) for K = 11  -->  96.67%
----------------------------------------


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_values, test_accuracies, marker='o', linestyle='-', color='g', linewidth=2, markersize=8)
plt.title('KNN Accuracy vs. K Value (Using load_iris)', fontsize=14)
plt.xlabel('Number of Neighbors (K)', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.xticks(k_values)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()


## Part 2 — scikit-learn baseline

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print("using sklearn")
print("=" * 60)

k_values = [1, 3, 5, 7, 11]

test_accuracies = []
train_accuracies = []

for k in k_values:
    knn_sklearn = KNeighborsClassifier(n_neighbors=k)
    knn_sklearn.fit(X_train_norm, y_train)
    
    y_pred_test = knn_sklearn.predict(X_test_norm)

    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_accuracies.append(test_accuracy)
    
    y_pred_train = knn_sklearn.predict(X_train_norm)

    train_accuracy = accuracy_score(y_train, y_pred_train)
    train_accuracies.append(train_accuracy)
    
    print(f"Accuracy in sklearn for K = {k:2d}  -->  {test_accuracy * 100:.2f}%")
    print(f"Accuracy(train) in sklearn for K = {k:2d}  -->  {train_accuracy * 100:.2f}%")
    print("-" * 50)  

using sklearn
Accuracy in sklearn for K =  1  -->  96.67%
Accuracy(train) in sklearn for K =  1  -->  100.00%
--------------------------------------------------
Accuracy in sklearn for K =  3  -->  93.33%
Accuracy(train) in sklearn for K =  3  -->  96.67%
--------------------------------------------------
Accuracy in sklearn for K =  5  -->  93.33%
Accuracy(train) in sklearn for K =  5  -->  96.67%
--------------------------------------------------
Accuracy in sklearn for K =  7  -->  93.33%
Accuracy(train) in sklearn for K =  7  -->  96.67%
--------------------------------------------------
Accuracy in sklearn for K = 11  -->  96.67%
Accuracy(train) in sklearn for K = 11  -->  96.67%
--------------------------------------------------


## Part 3 — Comparing distance metrics

In [15]:
print("\n Comparing distance metrics (5 = K)")
print("=" * 60)

models = {
    "Euclidean": KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    "Manhattan": KNeighborsClassifier(n_neighbors=5, metric='manhattan'),
    "Minkowski (p=3)": KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=3)
}

for name, model in models.items():
    model.fit(X_train_norm, y_train)
    
    y_pred = model.predict(X_test_norm)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"Distance metric : {name:<18} | Accuracy: {accuracy * 100:.2f}%")


 Comparing distance metrics (5 = K)
Distance metric : Euclidean          | Accuracy: 93.33%
Distance metric : Manhattan          | Accuracy: 96.67%
Distance metric : Minkowski (p=3)    | Accuracy: 93.33%


### Why Did Manhattan Distance Perform Better Here?

Manhattan distance doesn't square feature differences the way Euclidean distance does.
In the Iris feature space (petal/sepal length and width), a few samples show a sudden,
outlier-like shift in one feature due to natural variation.

Because Euclidean distance squares differences, it's very sensitive to large deviations
and penalizes them heavily, which can distort predictions. Manhattan distance penalizes
that same noise linearly instead, making it more robust — and in this experiment, it
classified more samples correctly as a result.
